# Классификация товаров по названиям

Полный пайплайн: анализ данных → предобработка → обучение модели → оценка → предсказание

# 1. Импорт библиотек

In [1]:
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, confusion_matrix
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

# 2. Загрузка данных

In [2]:
train_df = pd.read_csv('../data/raw/train_supervised_dataset.csv')
test_df = pd.read_csv('../data/raw/test_dataset.csv')
print(f'Обучающая выборка: {train_df.shape}')
print(f'Тестовая выборка: {test_df.shape}')

Обучающая выборка: (25000, 4)
Тестовая выборка: (5000, 2)


# 3. Исследование данных (EDA)

In [3]:
print('Колонки:', list(train_df.columns))
train_df.head()

Колонки: ['id', 'name', 'good', 'brand']


,id,name,good,brand
0,0,Petmax Бантик леопард с красн розой 2шт,бантик,petmax
1,1,87191 Бусы для елки шарики_87191,бусы,NaN
2,2,Футболка Piazza Italia WR011446881,футболка,piazza italia
3,3,7) YI572-03X-ONE ЗАКОЛКА ДЛЯ ВОЛОС ДЛЯ ДЕВОЧКИ,заколка,NaN
4,4,Одежда (вес) 1500,одежда,NaN


In [4]:
print(f'Уникальных категорий (good): {train_df["good"].nunique()}')
print('\nТоп-10 категорий:')
print(train_df['good'].value_counts().head(10))

Уникальных категорий (good): 2819

Топ-10 категорий:
good
брюки       339
пиво        327
вода        308
печенье     291
таблетки    271
сыр         266
чай         266
масло       247
напиток     232
носки       221
Name: count, dtype: int64


In [5]:
category_counts = train_df['good'].value_counts()
print(f'Самая частая: "{category_counts.index[0]}" ({category_counts.iloc[0]} Samples)')
print(f'Самая редкая: "{category_counts.index[-1]}" ({category_counts.iloc[-1]} Samples)')
print(f'Медианное количество: {category_counts.median():.0f}')
print(f'Заполнено брендов: {train_df["brand"].notna().sum()} из {len(train_df)}')
print(f'Уникальных брендов: {train_df["brand"].nunique()}')

Самая частая: "брюки" (339 Samples)
Самая редкая: "комплект установочный" (1 Samples)
Медианное количество: 2
Заполнено брендов: 16495 из 25000
Уникальных брендов: 6975


In [6]:
train_df['name_length'] = train_df['name'].str.len()
print(f'Мин. длина названия: {train_df["name_length"].min()} сим.')
print(f'Макс. длина названия: {train_df["name_length"].max()} сим.')
print(f'Средняя длина названия: {train_df["name_length"].mean():.1f} сим.')

Мин. длина названия: 5 сим.
Макс. длина названия: 128 сим.
Средняя длина названия: 41.8 сим.


In [7]:
train_df[['name', 'good', 'brand']].head(10)

,name,good,brand
0,Petmax Бантик леопард с красн розой 2шт,бантик,petmax
1,87191 Бусы для елки шарики_87191,бусы,NaN
2,Футболка Piazza Italia WR011446881,футболка,piazza italia
3,7) YI572-03X-ONE ЗАКОЛКА ДЛЯ ВОЛОС ДЛЯ ДЕВОЧКИ,заколка,NaN
4,Одежда (вес) 1500,одежда,NaN
5,РУСАЛОЧКА Губка ТРИО трехслой.3 шт ЛЮКС,губка,русалочка
6,"29-003 ОБОИ/ART/п.п. белые на флизе 1,06*10/ос...",обои,NaN
7,Квас Очаковский Пряная зима 2л,квас,очаковский
8,14 2456549180228 НБ маска+п/COSY/М-Ив,маска,cosy
9,Наконечник вилочный изолированный НВИ 2-6 (50ш...,наконечник,ekf


# 4. Предобработка — создание супер-категорий

In [8]:
def create_super_categories(df):
    category_mapping = {}
    categories = df['good'].unique()

    for category in categories:
        category_lower = str(category).lower()

        if any(word in category_lower for word in
               ['брюки', 'футболка', 'платье', 'кофта', 'рубашка', 'юбка', 'куртка', 'пальто', 'джемпер', 'шорты',
                'трусы', 'носки', 'кроссовки', 'белье', 'свитер', 'блузка', 'пиджак', 'жилет', 'майка', 'бант',
                'заколка', 'пуловер', 'блуза', 'фуфайка', 'сандал', 'одежда', 'кардиган', 'купальник']):
            category_mapping[category] = 'одежда'
        elif any(word in category_lower for word in
                 ['пиво', 'вода', 'молоко', 'хлеб', 'печенье', 'сыр', 'колбаса', 'сок', 'чай', 'масло', 'напиток',
                  'шоколад', 'мороженое', 'йогурт', 'корм', 'салат', 'конфеты', 'кофе', 'сахар', 'мука', 'консерв',
                  'колбас', 'мясо', 'рыба', 'овощ', 'фрукт', 'банан', 'яблоко', 'макарон', 'хлопь', 'коктейль', 'квас',
                  'рулет', 'круп', 'пицц', 'пельмен', 'драже', 'нектар', 'пирог', 'пломбир', 'перец', 'семен', 'огурц',
                  'булочк', 'десерт', 'сироп', 'икра', 'соль', 'капуста', 'батон']):
            category_mapping[category] = 'еда'
        elif any(word in category_lower for word in
                 ['батарейк', 'аккумулятор', 'зарядк', 'кабель', 'провод', 'розетк', 'выключатель']):
            category_mapping[category] = 'электроника'
        elif any(word in category_lower for word in
                 ['книга', 'учебник', 'тетрадь', 'блокнот', 'дневник', 'альбом', 'ручка', 'карандаш', 'бумага', 'краск',
                  'кисть', 'канцеляр', 'конструктор', 'маркер', 'резинка']):
            category_mapping[category] = 'канцелярия'
        elif any(word in category_lower for word in
                 ['труба', 'лак', 'краск', 'эмаль', 'герметик', 'пена', 'валик', 'угол', 'кран', 'шланг', 'смеситель']):
            category_mapping[category] = 'стройматериалы'
        elif any(word in category_lower for word in
                 ['таблетки', 'лекарств', 'витамины', 'медицин', 'аптечка', 'препарат', 'болеутол', 'здоровье',
                  'медицинск', 'шприц', 'лейкопластырь', 'прокладк', 'презерватив']):
            category_mapping[category] = 'здоровье'
        elif any(word in category_lower for word in
                 ['крем', 'шампунь', 'косметика', 'мыло', 'гель', 'лосьон', 'дезодорант', 'парфюм', 'духи', 'туалетн',
                  'гигиен', 'жидкост', 'салфетк', 'ароматизатор', 'кондиционер']):
            category_mapping[category] = 'косметика'
        elif any(word in category_lower for word in
                 ['мебель', 'посуда', 'ковер', 'шторы', 'бытов', 'лампа', 'светильник', 'интерьер', 'кухон', 'ванн',
                  'хозтовар', 'чехол', 'кашпо']):
            category_mapping[category] = 'дом'
        elif any(word in category_lower for word in ['сигарет', 'табак', 'никотин']):
            category_mapping[category] = 'табак'
        elif df[df['good'] == category].shape[0] > 25:
            category_mapping[category] = category
        else:
            category_mapping[category] = 'другое'

    df['super_category'] = df['good'].map(category_mapping)
    return df

In [9]:
train_processed = create_super_categories(train_df)

print(f'Уникальных супер-категорий: {train_processed["super_category"].nunique()}')
print('\nРаспределение супер-категорий:')
print(train_processed['super_category'].value_counts())

Уникальных супер-категорий: 97

Распределение супер-категорий:
super_category
другое          9036
еда             5143
одежда          2257
канцелярия       886
косметика        848
                ... 
яйцо              26
сапоги            26
пластырь          26
зубная паста      26
оливки            26
Name: count, Length: 97, dtype: int64


In [10]:
other_df = train_processed[train_processed['super_category'] == 'другое']
print(f'Объектов в "другое": {len(other_df)}')
print('\nТоп-20 категорий, попавших в "другое":')
print(other_df['good'].value_counts().head(20))

Объектов в "другое": 9036

Топ-20 категорий, попавших в "другое":
good
набор        25
топ          25
пленка       24
зажигалка    23
говядина     23
тройник      23
саморезы     23
подушка      23
точилка      23
фонарь       23
бинт         23
ключ         23
смазка       23
ластик       23
котлеты      23
блинчики     22
творожок     22
стакан       22
шар          22
муфта        22
Name: count, dtype: int64


In [11]:
all_words = []
for name in other_df['name'].dropna():
    words = str(name).lower().split()
    all_words.extend(words)
common_words = Counter(all_words).most_common(15)
print('Частые слова в "другое":')
for word, count in common_words:
    print(f'  {word}: {count}')

Частые слова в "другое":
  с: 913
  для: 572
  шт: 383
  в: 356
  1: 246
  и: 227
  (шт): 199
  на: 174
  2: 170
  г: 168
  из: 158
  набор: 150
  -: 147
  шт.: 130
  мм: 112


# 5. Разделение на обучающую и тестовую выборки

In [12]:
X = train_processed['name']
y = train_processed['super_category']

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Обучающая выборка: {x_train.shape[0]} samples')
print(f'Тестовая выборка: {x_test.shape[0]} samples')

Обучающая выборка: 20000 samples
Тестовая выборка: 5000 samples


# 6. Векторизация TF-IDF

In [13]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.8
)

x_train_tfidf = vectorizer.fit_transform(x_train)
x_test_tfidf = vectorizer.transform(x_test)

print(f'Размерность признаков: {x_train_tfidf.shape}')
print(f'Размер словаря: {len(vectorizer.vocabulary_)}')

Размерность признаков: (20000, 5000)
Размер словаря: 5000


# 7. Обучение модели (Logistic Regression)

In [14]:
classifier = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced'
)

classifier.fit(x_train_tfidf, y_train)
y_pred = classifier.predict(x_test_tfidf)

print(f'Количество классов: {len(classifier.classes_)}')

Количество классов: 97


## 8. Оценка качества

In [15]:
print(f'F1-score (micro):   {f1_score(y_test, y_pred, average="micro", zero_division=0):.4f}')
print(f'F1-score (macro):   {f1_score(y_test, y_pred, average="macro", zero_division=0):.4f}')
print(f'F1-score (weighted): {f1_score(y_test, y_pred, average="weighted", zero_division=0):.4f}')

F1-score (micro):   0.8050
F1-score (macro):   0.8758
F1-score (weighted): 0.8161


In [16]:
print('Отчет по классификации (топ-10 категорий):')
top_categories = y_test.value_counts().head(10).index
y_test_top = y_test[y_test.isin(top_categories)]
y_pred_top = y_pred[pd.Series(y_test).isin(top_categories)]
print(classification_report(y_test_top, y_pred_top, zero_division=0))

Отчет по классификации (топ-10 категорий):
                precision    recall  f1-score   support

      антифриз       0.00      0.00      0.00         0
        арахис       0.00      0.00      0.00         0
       бальзам       0.00      0.00      0.00         0
         билет       0.00      0.00      0.00         0
          вино       0.00      0.00      0.00         0
         водка       0.00      0.00      0.00         0
        джинсы       0.00      0.00      0.00         0
           дом       0.91      0.92      0.91        64
        другое       0.95      0.58      0.72      1807
           еда       0.82      0.91      0.86      1029
      здоровье       0.81      0.94      0.87       101
  зубная паста       0.00      0.00      0.00         0
       игрушка       0.00      0.00      0.00         0
    канцелярия       0.80      0.88      0.84       177
         капли       0.00      0.00      0.00         0
       капсулы       0.00      0.00      0.00         0
    

## 9. Сохранение модели

In [17]:
os.makedirs('../models', exist_ok=True)
joblib.dump(vectorizer, '../models/vectorizer.joblib')
joblib.dump(classifier, '../models/classifier.joblib')
print('Модели сохранены в папку models/')

Модели сохранены в папку models/


## 10. Загрузка модели и предсказание на тестовых данных

In [18]:
loaded_vectorizer = joblib.load('../models/vectorizer.joblib')
loaded_classifier = joblib.load('../models/classifier.joblib')
print('Модели загружены')

Модели загружены


In [19]:
test_tfidf = loaded_vectorizer.transform(test_df['name'])
test_predictions = loaded_classifier.predict(test_tfidf)

print('Примеры предсказаний:')
pd.DataFrame({'name': test_df['name'].head(10), 'predicted': test_predictions[:10]})

Примеры предсказаний:


,name,predicted
0,"469-210 ЕРМАК Клей универсальный, 15мл, блистер",клей
1,Торт СЛАДУШКА Зимняя вишня 700г,торт
2,"Смеситель ""CALORIE"" 1023 А06 д/кухни",стройматериалы
3,Лимон 50гр БАР,еда
4,"Коньяк САРАДЖИШВИЛИ 5 лет 0,5л Грузия",коньяк
5,"Born Pretty, Пластина для стемпинга BP-L011 Te...",канцелярия
6,Рис Chicken Beer с гуляшом 250 гр,рис
7,Труба Tebo п/п D-32 PN20 4м стекловолокно хлыст,стройматериалы
8,Грандорф Консервы для собак Куропатка с Индейк...,другое
9,"1) 000893|Одеяло ""Автотепло"" №8",другое


## 11. Создание файла submission.csv

In [20]:
submission = pd.DataFrame({
    'id': test_df['id'],
    'good': test_predictions,
    'brand': ''
})

submission.to_csv('../submission.csv', index=False)
print(f'Сохранено {len(submission)} строк в submission.csv')
submission.head(10)

Сохранено 5000 строк в submission.csv


,id,good,brand
0,0,клей,
1,1,торт,
2,2,стройматериалы,
3,3,еда,
4,4,коньяк,
5,5,канцелярия,
6,6,рис,
7,7,стройматериалы,
8,8,другое,
9,9,другое,
